# Phase 8: Exploratory Data Analysis (EDA) & Outlier Detection
## IBM Data Analyst Professional Certificate Capstone Project

**Author**: Raghavendra Golla  
**Theme**: Rigorous Distribution Analysis, IQR Outlier Detection & Correlation Profiling  

---

### 1. Executive Context & Objectives
In this notebook, we perform formal Exploratory Data Analysis (EDA) to uncover the distributional properties, skewness, and anomalies in developer compensation, working hours, and career demographics.

#### Core Business & Analytical Questions:
1. **Distribution of Compensation**: Is developer compensation normally distributed, and how severe is the skewness?
2. **Outlier Detection via IQR**: How many high-earning outliers distort the mean, and what is the true median salary when outliers are filtered?
3. **Demographics & Experience**: How is the developer workforce distributed across age brackets?
4. **Correlation Analysis**: Do longer work weeks or more code review hours correlate with higher compensation?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set publication aesthetic
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Load cleaned survey dataset
CLEAN_PATH = os.path.join("..", "data", "processed", "cleaned_survey_data.csv")
if not os.path.exists(CLEAN_PATH):
    CLEAN_PATH = os.path.join("data", "processed", "cleaned_survey_data.csv")

df = pd.read_csv(CLEAN_PATH)
print(f"Cleaned dataset successfully loaded: {len(df):,} rows.")

---
### 2. Distribution Analysis of Developer Compensation (`ConvertedComp`)
We analyze the shape of the distribution of annual compensation in USD.

In [ ]:
comp_series = df['ConvertedComp'].dropna()
print(f"Valid compensation responses: {len(comp_series):,}")
print(f"Mean Compensation:   ${comp_series.mean():,.2f}")
print(f"Median Compensation: ${comp_series.median():,.2f}")
print(f"Std Deviation:       ${comp_series.std():,.2f}")
print(f"Skewness:            {comp_series.skew():.2f} (Highly right-skewed)")

---
### 3. Outlier Identification Using the Interquartile Range (IQR) Method
Because compensation is heavily right-skewed, the standard parametric rule (mean $\pm$ 3 standard deviations) is invalid. We use the robust **Tukey's Fences (IQR)** method:
$$\text{IQR} = Q3 - Q1$$
$$\text{Lower Bound} = Q1 - 1.5 \times \text{IQR}$$
$$\text{Upper Bound} = Q3 + 1.5 \times \text{IQR}$$

In [ ]:
Q1 = comp_series.quantile(0.25)
Q3 = comp_series.quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers_upper = comp_series[comp_series > upper_fence]
outliers_lower = comp_series[comp_series < lower_fence]
non_outliers = comp_series[(comp_series >= lower_fence) & (comp_series <= upper_fence)]

print(f"Q1 (25th percentile):    ${Q1:,.2f}")
print(f"Q3 (75th percentile):    ${Q3:,.2f}")
print(f"IQR:                     ${IQR:,.2f}")
print(f"Lower Bound (Fence):     ${lower_fence:,.2f}")
print(f"Upper Bound (Fence):     ${upper_fence:,.2f}")
print(f"Number of Upper Outliers: {len(outliers_upper):,} ({len(outliers_upper)/len(comp_series)*100:.2f}%)")
print(f"Number of Lower Outliers: {len(outliers_lower):,}")
print(f"\n-- Post-Outlier Removal Metrics --")
print(f"Mean after removing outliers:   ${non_outliers.mean():,.2f}")
print(f"Median after removing outliers: ${non_outliers.median():,.2f}")

---
### 4. Demographic Profiling: Age Distribution & Outliers
We inspect the age distribution across the global developer population.

In [ ]:
age_series = df['Age'].dropna()
age_q1 = age_series.quantile(0.25)
age_q3 = age_series.quantile(0.75)
age_iqr = age_q3 - age_q1
age_lower = age_q1 - 1.5 * age_iqr
age_upper = age_q3 + 1.5 * age_iqr

print(f"Age Summary:")
print(f"  Mean:   {age_series.mean():.1f} years")
print(f"  Median: {age_series.median():.1f} years")
print(f"  Min:    {age_series.min():.0f} years")
print(f"  Max:    {age_series.max():.0f} years")
print(f"  Outliers below Q1 - 1.5*IQR ({age_lower:.1f}): {(age_series < age_lower).sum()}")
print(f"  Outliers above Q3 + 1.5*IQR ({age_upper:.1f}): {(age_series > age_upper).sum()}")

---
### 5. Correlation Analysis: Numerical Variables
We evaluate pairwise linear (Pearson) and monotonic (Spearman) relationships between `Age`, `WorkWeekHrs`, `CodeRevHrs`, and `ConvertedComp`.

In [ ]:
num_cols = ['Age', 'WorkWeekHrs', 'CodeRevHrs', 'ConvertedComp']
# Filter impossible work hours for reliable correlation
df_corr_subset = df[df['WorkWeekHrs'].between(10, 100)][num_cols].dropna()

pearson_corr = df_corr_subset.corr(method='pearson')
spearman_corr = df_corr_subset.corr(method='spearman')

print("Pearson Correlation Matrix (Linear Relationships):")
display(pearson_corr)

print("\nSpearman Rank Correlation Matrix (Monotonic Relationships):")
display(spearman_corr)

---
### 6. Key Findings & Strategic Insights

| Variable | Finding | Business Meaning |
| :--- | :--- | :--- |
| **`ConvertedComp` Skewness** | Skewness of 5.38; Mean is $131,334 while Median is $57,744. | Leadership must NEVER use average salary to estimate hiring budgets; median is the only reliable benchmark. |
| **Compensation Outliers** | 879 respondents earn above the upper fence of $209,698 (up to $2M). | When outliers are excluded, the normalized mean drops from $131,334 to $59,883, closely aligning with the $52,704 median. |
| **Age Profile** | Median age is 29.0 years; 75% of respondents are under 35. | High concentration of early-to-mid career professionals seeking career progression and technical mentorship. |
| **Hours vs. Pay Correlation** | Weak correlation ($r = 0.05$) between `WorkWeekHrs` and compensation. | Working longer hours does not translate to higher market pay; skill scarcity and role seniority drive compensation, not raw overtime. |

> *"In Phase 8, I used the IQR method to detect 879 high-earning salary outliers beyond the $209,698 fence. This revealed that the $131k mean was inflated by extreme executive salaries up to $2M. Trimming outliers normalized the mean to $59.8k, proving that the true market baseline is ~$57k-$60k. Furthermore, correlation analysis proved that compensation has almost zero correlation with overtime work hours (r=0.05), emphasizing that our consulting clients must reward technical specialization rather than raw hours worked."*